In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
#bulid the spark session
spark = SparkSession.builder \
    .appName("ASS") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()  
#use exist one if have , if don have create new one

In [3]:
from pyspark.sql.functions import col, sum, when , trim, lower,  avg, min, max , upper


## read historical file

In [4]:
df = spark.read.csv("historical_data.csv", header=True, inferSchema=True)  
                                                        #inferSchema here spark will guese the data type, else all will threat as string
df.show()

+-------+------------------+-------------------+-----------------------+-------------------+--------------------+--------------------------+---------------------------+--------------------+--------------------+-----------------+-------------+---------------+-------------------+--------+--------------+-------------------+-------------------+-------------------+
|     ID|Delivery_person_ID|Delivery_person_Age|Delivery_person_Ratings|Restaurant_latitude|Restaurant_longitude|Delivery_location_latitude|Delivery_location_longitude|   Weatherconditions|Road_traffic_density|Vehicle_condition|Type_of_order|Type_of_vehicle|multiple_deliveries|Festival|          City|     Order_DateTime|    Picked_Datetime| Delivered_Datetime|
+-------+------------------+-------------------+-----------------------+-------------------+--------------------+--------------------------+---------------------------+--------------------+--------------------+-----------------+-------------+---------------+----------------

## show schema (data type checking)

In [5]:
df.printSchema()
print(df.columns)

root
 |-- ID: string (nullable = true)
 |-- Delivery_person_ID: string (nullable = true)
 |-- Delivery_person_Age: double (nullable = true)
 |-- Delivery_person_Ratings: double (nullable = true)
 |-- Restaurant_latitude: double (nullable = true)
 |-- Restaurant_longitude: double (nullable = true)
 |-- Delivery_location_latitude: double (nullable = true)
 |-- Delivery_location_longitude: double (nullable = true)
 |-- Weatherconditions: string (nullable = true)
 |-- Road_traffic_density: string (nullable = true)
 |-- Vehicle_condition: integer (nullable = true)
 |-- Type_of_order: string (nullable = true)
 |-- Type_of_vehicle: string (nullable = true)
 |-- multiple_deliveries: double (nullable = true)
 |-- Festival: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Order_DateTime: timestamp (nullable = true)
 |-- Picked_Datetime: timestamp (nullable = true)
 |-- Delivered_Datetime: timestamp (nullable = true)

['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_

In [6]:
df.select(col("Delivery_person_ID")).show()
df.select(col("ID")).show()

numeric_categorical = ["Delivery_person_Age","Delivery_person_Ratings","Restaurant_latitude","Restaurant_longitude","Delivery_location_latitude","Delivery_location_longitude"]

for c in numeric_categorical:
    df.select(col(c)).show()

+------------------+
|Delivery_person_ID|
+------------------+
|    BHPRES15DEL03 |
|   LUDHRES11DEL02 |
|    KOCRES05DEL03 |
|    DEHRES04DEL03 |
|    AGRRES08DEL01 |
|   LUDHRES18DEL02 |
|    ALHRES20DEL01 |
|    ALHRES17DEL02 |
|    KOCRES12DEL02 |
|    GOARES09DEL02 |
|   AURGRES02DEL01 |
|    AGRRES17DEL01 |
|    ALHRES06DEL02 |
|    KOCRES20DEL02 |
|    AGRRES06DEL02 |
|    ALHRES04DEL02 |
|   AURGRES15DEL02 |
|    BHPRES13DEL01 |
|    KOLRES14DEL01 |
|   AGRRES010DEL01 |
+------------------+
only showing top 20 rows
+-------+
|     ID|
+-------+
|0xd152 |
|0xc401 |
|0xd300 |
|0xc614 |
|0xcb30 |
|0xceea |
|0xc3ff |
|0xd8ff |
|0xd8c5 |
|0xdaa5 |
|0xcc98 |
|0xd2bf |
|0xcfd0 |
|0xcf03 |
|0xe1cc |
|0xd39b |
|0xd998 |
|0xcf11 |
|0xd489 |
|0xdaa2 |
+-------+
only showing top 20 rows
+-------------------+
|Delivery_person_Age|
+-------------------+
|               38.0|
|               25.0|
|               39.0|
|               36.0|
|               22.0|
|               33.0|
|       

## show unique value for categorical data 

In [7]:
df_show = ["Weatherconditions",
           "Road_traffic_density",
           "Vehicle_condition",
           "Type_of_order",
           "Type_of_vehicle",
           "multiple_deliveries",
           "Festival",
           "City"]
for c in df_show:
    print(f"Unique values in column: {c}")
    df.select(c).distinct().show(truncate=False)

Unique values in column: Weatherconditions
+---------------------+
|Weatherconditions    |
+---------------------+
|conditions Sandstorms|
|conditions Fog       |
|conditions Windy     |
|conditions Sunny     |
|conditions NaN       |
|conditions Cloudy    |
|conditions Stormy    |
+---------------------+

Unique values in column: Road_traffic_density
+--------------------+
|Road_traffic_density|
+--------------------+
|Medium              |
|NaN                 |
|Low                 |
|High                |
|Jam                 |
+--------------------+

Unique values in column: Vehicle_condition
+-----------------+
|Vehicle_condition|
+-----------------+
|1                |
|3                |
|2                |
|0                |
+-----------------+

Unique values in column: Type_of_order
+-------------+
|Type_of_order|
+-------------+
|Snack        |
|Buffet       |
|Drinks       |
|Meal         |
+-------------+

Unique values in column: Type_of_vehicle
+-----------------+
|Type

## standardize categorical data (nominal categorical and ordinal categorical, not include ordinal numeric)

In [8]:

categorical_cols = [
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City"
]

for c in categorical_cols:
    df = df.withColumn(c, trim(col(c)))   #remove space
    df = df.withColumn(c, lower(col(c))) 
    df.select(col(c)).show()
#df.select(col("City")).show()

+--------------------+
|   Weatherconditions|
+--------------------+
|      conditions fog|
|conditions sandst...|
|    conditions windy|
|    conditions sunny|
|    conditions windy|
|    conditions sunny|
|   conditions stormy|
|    conditions sunny|
|    conditions sunny|
|    conditions sunny|
|    conditions windy|
|conditions sandst...|
|   conditions cloudy|
|      conditions fog|
|   conditions stormy|
|conditions sandst...|
|   conditions cloudy|
|   conditions cloudy|
|   conditions stormy|
|      conditions fog|
+--------------------+
only showing top 20 rows
+--------------------+
|Road_traffic_density|
+--------------------+
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|                 low|
|        

In [9]:
df = df.withColumn("Delivery_person_ID",trim(col("Delivery_person_ID")))  # need upper case?
df = df.withColumn("Delivery_person_ID",upper(col("Delivery_person_ID")))

df.select(col("Delivery_person_ID")).show()

+------------------+
|Delivery_person_ID|
+------------------+
|     BHPRES15DEL03|
|    LUDHRES11DEL02|
|     KOCRES05DEL03|
|     DEHRES04DEL03|
|     AGRRES08DEL01|
|    LUDHRES18DEL02|
|     ALHRES20DEL01|
|     ALHRES17DEL02|
|     KOCRES12DEL02|
|     GOARES09DEL02|
|    AURGRES02DEL01|
|     AGRRES17DEL01|
|     ALHRES06DEL02|
|     KOCRES20DEL02|
|     AGRRES06DEL02|
|     ALHRES04DEL02|
|    AURGRES15DEL02|
|     BHPRES13DEL01|
|     KOLRES14DEL01|
|    AGRRES010DEL01|
+------------------+
only showing top 20 rows


## remove duplicate

In [10]:
# count before removing duplicates
before_count = df.count()

# remove duplicate
df = df.dropDuplicates()

# count after removing duplicates
after_count = df.count()

removed_duplicated_count = before_count - after_count

print("Rows before removing duplicates:", before_count)
print("Rows after removing duplicates:", after_count)
print("Duplicate rows removed:", removed_duplicated_count)



Rows before removing duplicates: 17097
Rows after removing duplicates: 17097
Duplicate rows removed: 0


## show missing data

In [11]:

all_data = ["ID",
            "Delivery_person_ID",
            "Delivery_person_Age",
            "Delivery_person_Ratings",
            "Restaurant_latitude","Restaurant_longitude",
            "Delivery_location_latitude","Delivery_location_longitude",
            #"Weatherconditions",
            "Road_traffic_density",
            "Vehicle_condition",
            "Type_of_order",
            "Type_of_vehicle",
            "multiple_deliveries",
            "Festival",
            "City",
            "Order_DateTime",
            "Picked_Datetime","Delivered_Datetime"]

for c in all_data:
    missing_data =  df.filter(
                        col(c).isNull() |  #check for python null type and spark none type only
                        (trim(col(c).cast("string")) == "") |
                        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
                    ).count()
    print(c)
    print("total number of missing data:",missing_data)
    print()

#since inside missing value is condition NaN 
missing_data = df.filter(
    lower(trim(col("Weatherconditions"))).contains("nan")
).count()
print("Weather Condition")
print("total number of missing data:",missing_data)

ID
total number of missing data: 0

Delivery_person_ID
total number of missing data: 0

Delivery_person_Age
total number of missing data: 657

Delivery_person_Ratings
total number of missing data: 727

Restaurant_latitude
total number of missing data: 0

Restaurant_longitude
total number of missing data: 0

Delivery_location_latitude
total number of missing data: 0

Delivery_location_longitude
total number of missing data: 0

Road_traffic_density
total number of missing data: 196

Vehicle_condition
total number of missing data: 0

Type_of_order
total number of missing data: 0

Type_of_vehicle
total number of missing data: 0

multiple_deliveries
total number of missing data: 364

Festival
total number of missing data: 78

City
total number of missing data: 426

Order_DateTime
total number of missing data: 653

Picked_Datetime
total number of missing data: 0

Delivered_Datetime
total number of missing data: 0

Weather Condition
total number of missing data: 215


## remove row data for important variable that missing

In [12]:
# direct remove for important variable
df_bfr_remove = df.count()

critical_cols = [
    "Delivery_person_ID",
    "Order_DateTime",
    "Picked_Datetime",
    "Delivered_Datetime",
    "Delivery_person_Ratings"
]

for c in critical_cols:
    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c)) == "") |
            (lower(trim(col(c))).isin("null", "na", "n/a", "nan")),
            None
        ).otherwise(col(c))
    )

df = df.na.drop(subset=critical_cols)
df_after_remove = df.count()
print("Rows before removing missing value:", df_bfr_remove)
print("Rows after removing missing value:", df_after_remove)

#Delivery_person_Age ,Vehicle_condition,multiple_deliveries

# since Restaurant_latitude, Restaurant_longitude, Delivery_location_latitude, Delivery_location_longitude no used 
# so just leave it when got missing value to prevet too many information dissapear 

#Picked_Datetime,Delivered_Datetime

Rows before removing missing value: 17097
Rows after removing missing value: 16336


## replace all missing categorical data with "unknown"

In [13]:
non_critical_category = ["Road_traffic_density", "Type_of_order", "Type_of_vehicle", "Festival", "City"]

for c in non_critical_category:
    
    count_bfr = df.filter(
        col(c).isNull() |
        (trim(col(c)) == "") |
        (lower(trim(col(c))).isin("null", "na", "n/a", "nan"))
    ).count()
    
    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c)) == "") |
            (lower(trim(col(c))).isin("null", "na", "n/a", "nan")),
            "Unknown"
        ).otherwise(col(c))
    )

    count_after = df.filter(
        col(c).isNull() |
        (trim(col(c)) == "") |
        (lower(trim(col(c))).isin("null", "na", "n/a", "nan"))
    ).count()

    print(c)
    print("Before count for inconsistent missing data :", count_bfr)
    print("After count for inconsistent missing data (all replaced by unknown) :", count_after)


print("row count:", df.count())

Road_traffic_density
Before count for inconsistent missing data : 0
After count for inconsistent missing data (all replaced by unknown) : 0
Type_of_order
Before count for inconsistent missing data : 0
After count for inconsistent missing data (all replaced by unknown) : 0
Type_of_vehicle
Before count for inconsistent missing data : 0
After count for inconsistent missing data (all replaced by unknown) : 0
Festival
Before count for inconsistent missing data : 73
After count for inconsistent missing data (all replaced by unknown) : 0
City
Before count for inconsistent missing data : 399
After count for inconsistent missing data (all replaced by unknown) : 0
row count: 16336


In [14]:
from pyspark.sql.functions import col, trim, lower, when

count_bfr = df.filter(
    (trim(col("Weatherconditions")) == "") |
    (lower(trim(col("Weatherconditions"))).contains("nan"))
).count()

df = df.withColumn(
    "Weatherconditions",
    when(
        (trim(col("Weatherconditions")) == "") |
        (lower(trim(col("Weatherconditions"))).contains("nan")),
        "Unknown"
    ).otherwise(col("Weatherconditions"))
)

count_after = df.filter(
    (trim(col("Weatherconditions")) == "") |
    (lower(trim(col("Weatherconditions"))).contains("nan"))
).count()

print(c)
print("Before count for inconsistent missing data :", count_bfr)
print("After count for inconsistent missing data (all replaced by unknown) :", count_after)

City
Before count for inconsistent missing data : 0
After count for inconsistent missing data (all replaced by unknown) : 0


## replace all missing ordinary data with median 

In [15]:
# since cannot replace by unknown so just replace by median

ordinary_data = ["multiple_deliveries","Vehicle_condition"]

for c in ordinary_data:
    
    missing_data_bfr =  df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()


    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == "") |
            (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan")),
            None
        ).otherwise(col(c))
    
    )
    q1, median_val, q3 = df.filter(col(c).isNotNull()).approxQuantile(c, [0.25, 0.5, 0.75], 0.01)

    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == "") |
            (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan")),
            median_val
        ).otherwise(col(c))
    )
    

    missing_data_after = df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()
    

    print ("Coloumn: ", c)
    print ("missing data before replace: ", missing_data_bfr)
    print ("missing data after replace", missing_data_after)
    
print("row count:", df.count())


Coloumn:  multiple_deliveries
missing data before replace:  340
missing data after replace 0
Coloumn:  Vehicle_condition
missing data before replace:  0
missing data after replace 0
row count: 16336


## replace all numerical data with mean

In [16]:


mean_val = df.select(avg("Delivery_person_Age")).collect()[0][0]
q1, median_val, q3 = df.approxQuantile("Delivery_person_Age", [0.25, 0.5, 0.75], 0.01)
min_val = df.select(min("Delivery_person_Age")).collect()[0][0]
max_val = df.select(max("Delivery_person_Age")).collect()[0][0]

print("Delivery_person_Age")
print("Min:", min_val)
print("Q1:", q1)
print("Median:", median_val)
print("Q3:", q3)
print("Mean:", mean_val)
print("Max:", max_val)

# since median is close to mean, use mean to handle missing values
# mean_val = round(mean_val, 2)
# count missing before
missing_before = df.filter(
    col("Delivery_person_Age").isNull() |
    (trim(col("Delivery_person_Age").cast("string")) == "") |
    (lower(trim(col("Delivery_person_Age").cast("string"))).isin("null", "na", "n/a", "nan"))
).count()

# replace missing with mean
df = df.withColumn(
    "Delivery_person_Age",
    when(
        col("Delivery_person_Age").isNull() |
        (trim(col("Delivery_person_Age").cast("string")) == "") |
        (lower(trim(col("Delivery_person_Age").cast("string"))).isin("null", "na", "n/a", "nan")),
        mean_val
    ).otherwise(col("Delivery_person_Age"))
)

# count missing after
missing_after = df.filter(
    col("Delivery_person_Age").isNull() |
    (trim(col("Delivery_person_Age").cast("string")) == "") |
    (lower(trim(col("Delivery_person_Age").cast("string"))).isin("null", "na", "n/a", "nan"))
).count()

print()
print("Mean used for replace:", mean_val)
print("Missing before:", missing_before)
print("Missing after:", missing_after)

print("row count:", df.count())

Delivery_person_Age
Min: 20.0
Q1: 25.0
Median: 29.0
Q3: 34.0
Mean: 29.575538687561213
Max: 39.0

Mean used for replace: 29.575538687561213
Missing before: 0
Missing after: 0
row count: 16336


## check for non-critical data (scare remove too many data if directly remove all invalid )  now handle by remove all first not sure

In [17]:
location_coordinates = ["Restaurant_latitude","Restaurant_longitude",
                     "Delivery_location_latitude","Delivery_location_longitude"]
    
for c in location_coordinates:
    
    missing_data_count_bfr = df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()
    
    df = df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c)) == "") |
            (lower(trim(col(c))).isin("null", "na", "n/a", "nan")),
            None
        ).otherwise(col(c))
    )

    missing_data_count_after = df.filter(
        col(c).isNull() |
        (trim(col(c).cast("string")) == "") |
        (lower(trim(col(c).cast("string"))).isin("null", "na", "n/a", "nan"))
    ).count()

    print(c)
    print("total number of missing data before cleaning:", missing_data_count_bfr)
    print("total number of missing data after cleaning:", missing_data_count_after)
    print()

df = df.na.drop(subset=critical_cols)

print("row count:", df.count())

Restaurant_latitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

Restaurant_longitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

Delivery_location_latitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

Delivery_location_longitude
total number of missing data before cleaning: 0
total number of missing data after cleaning: 0

row count: 16336


# range check

Missing values were imputed because the remaining fields in the record could still be valid and useful. However, rows containing invalid values were removed, since such values indicate data quality errors and reduce confidence in the reliability of the entire record. values outside predefined valid ranges are treated as invalid and the affected records are filtered out.

In [18]:
vehicle_out_of_range_couunt_bfr = df.filter((col("Vehicle_condition") < 0) | (col("Vehicle_condition") > 3)).count()
rating_out_of_range_count_bfr = df.filter((col("Delivery_person_Ratings") < 1) | (col("Delivery_person_Ratings") > 5)).count()
age_out_of_range_count_bfr = df.filter((col("Delivery_person_Age") < 18) | (col("Delivery_person_Age") > 65)).count()
multiple_deliveries_out_of_range_count_bfr = df.filter(col("multiple_deliveries") < 0).count()

# remove rows that contain any out-of-range value
df = df.filter(
    (col("Vehicle_condition") >= 0) & (col("Vehicle_condition") <= 3) &
    (col("Delivery_person_Ratings") >= 1) & (col("Delivery_person_Ratings") <= 5) &
    (col("Delivery_person_Age") >= 18) & (col("Delivery_person_Age") <= 65) &
    (col("multiple_deliveries") >= 0)
)

vehicle_out_of_range_couunt_after = df.filter((col("Vehicle_condition") < 0) | (col("Vehicle_condition") > 3)).count()
rating_out_of_range_count_after = df.filter((col("Delivery_person_Ratings") < 1) | (col("Delivery_person_Ratings") > 5)).count()
age_out_of_range_count_after = df.filter((col("Delivery_person_Age") < 18) | (col("Delivery_person_Age") > 65)).count()
multiple_deliveries_out_of_range_count_after = df.filter(col("multiple_deliveries") < 0).count()


print ("number vehicle condition that out of range before : ",vehicle_out_of_range_couunt_bfr)
print ("number vehicle condition that out of range after: ",vehicle_out_of_range_couunt_after)
print()
print ("number delivery person ratings that out of range before : ",rating_out_of_range_count_bfr)
print ("number delivery person ratings that out of range after : ",rating_out_of_range_count_after)
print()
print ("number delivery person age that out of range before : ",age_out_of_range_count_bfr)
print ("number delivery person age that out of range after : ",age_out_of_range_count_after)
print()
print ("number multiple deliveries that out of range before : ",multiple_deliveries_out_of_range_count_bfr)
print ("number multiple deliveries that out of range after : ",multiple_deliveries_out_of_range_count_after)

print("row count:", df.count())

number vehicle condition that out of range before :  0
number vehicle condition that out of range after:  0

number delivery person ratings that out of range before :  0
number delivery person ratings that out of range after :  0

number delivery person age that out of range before :  0
number delivery person age that out of range after :  0

number multiple deliveries that out of range before :  0
number multiple deliveries that out of range after :  0
row count: 16336


## timestamp check

In [19]:
invalid_timestamp_order_before = df.filter(
    (col("Picked_Datetime") < col("Order_DateTime")) |
    (col("Delivered_Datetime") < col("Picked_Datetime"))
).count()

invalid_timestamp_order_before_list = df.filter(
    (col("Picked_Datetime") < col("Order_DateTime")) |
    (col("Delivered_Datetime") < col("Picked_Datetime"))
)


df = df.filter(
    (col("Picked_Datetime") >= col("Order_DateTime")) &
    (col("Delivered_Datetime") >= col("Picked_Datetime"))
)

invalid_timestamp_order_after = df.filter(
    (col("Picked_Datetime") < col("Order_DateTime")) |
    (col("Delivered_Datetime") < col("Picked_Datetime"))
).count()


# show some invalid examples
print("Example invalid timestamp rows:")
invalid_timestamp_order_before_list.select(
    "Order_DateTime",
    "Picked_Datetime",
    "Delivered_Datetime"
).show(10, False)
print("invalid timestamp order before remove:", invalid_timestamp_order_before)
print("invalid timestamp order after remove:", invalid_timestamp_order_after)
print("row count:", df.count())

Example invalid timestamp rows:
+-------------------+-------------------+-------------------+
|Order_DateTime     |Picked_Datetime    |Delivered_Datetime |
+-------------------+-------------------+-------------------+
|2022-02-11 23:50:00|2022-02-11 00:00:00|2022-02-11 00:30:00|
|2022-02-11 23:55:00|2022-02-11 00:00:00|2022-02-11 00:29:00|
|2022-02-17 23:45:00|2022-02-17 00:00:00|2022-02-17 00:24:00|
|2022-03-03 23:45:00|2022-03-03 00:00:00|2022-03-03 00:33:00|
|2022-02-14 23:50:00|2022-02-14 00:00:00|2022-02-14 00:37:00|
|2022-03-03 23:45:00|2022-03-03 00:00:00|2022-03-03 00:24:00|
|2022-02-18 23:55:00|2022-02-18 00:10:00|2022-02-18 00:53:00|
|2022-03-04 23:45:00|2022-03-04 00:00:00|2022-03-04 00:37:00|
|2022-02-14 23:55:00|2022-02-14 00:00:00|2022-02-14 00:39:00|
|2022-03-01 23:45:00|2022-03-01 00:00:00|2022-03-01 00:29:00|
+-------------------+-------------------+-------------------+
only showing top 10 rows
invalid timestamp order before remove: 325
invalid timestamp order after re

## coordinate check

In [20]:
# before remove
restaurant_lat_out_of_range_before = df.filter(
    (col("Restaurant_latitude") < -90) | (col("Restaurant_latitude") > 90)
).count()

restaurant_long_out_of_range_before = df.filter(
    (col("Restaurant_longitude") < -180) | (col("Restaurant_longitude") > 180)
).count()

delivery_lat_out_of_range_before = df.filter(
    (col("Delivery_location_latitude") < -90) | (col("Delivery_location_latitude") > 90)
).count()

delivery_long_out_of_range_before = df.filter(
    (col("Delivery_location_longitude") < -180) | (col("Delivery_location_longitude") > 180)
).count()

# remove rows with invalid coordinates
df = df.filter(
    (col("Restaurant_latitude") >= -90) & (col("Restaurant_latitude") <= 90) &
    (col("Restaurant_longitude") >= -180) & (col("Restaurant_longitude") <= 180) &
    (col("Delivery_location_latitude") >= -90) & (col("Delivery_location_latitude") <= 90) &
    (col("Delivery_location_longitude") >= -180) & (col("Delivery_location_longitude") <= 180)
)

# after remove
restaurant_lat_out_of_range_after = df.filter(
    (col("Restaurant_latitude") < -90) | (col("Restaurant_latitude") > 90)
).count()

restaurant_long_out_of_range_after = df.filter(
    (col("Restaurant_longitude") < -180) | (col("Restaurant_longitude") > 180)
).count()

delivery_lat_out_of_range_after = df.filter(
    (col("Delivery_location_latitude") < -90) | (col("Delivery_location_latitude") > 90)
).count()

delivery_long_out_of_range_after = df.filter(
    (col("Delivery_location_longitude") < -180) | (col("Delivery_location_longitude") > 180)
).count()

print("Before removal")
print("restaurant latitude out of range:", restaurant_lat_out_of_range_before)
print("restaurant longitude out of range:", restaurant_long_out_of_range_before)
print("delivery latitude out of range:", delivery_lat_out_of_range_before)
print("delivery longitude out of range:", delivery_long_out_of_range_before)
print()

print("After removal")
print("restaurant latitude out of range:", restaurant_lat_out_of_range_after)
print("restaurant longitude out of range:", restaurant_long_out_of_range_after)
print("delivery latitude out of range:", delivery_lat_out_of_range_after)
print("delivery longitude out of range:", delivery_long_out_of_range_after)

Before removal
restaurant latitude out of range: 0
restaurant longitude out of range: 0
delivery latitude out of range: 0
delivery longitude out of range: 0

After removal
restaurant latitude out of range: 0
restaurant longitude out of range: 0
delivery latitude out of range: 0
delivery longitude out of range: 0


## create new variable

In [21]:
from pyspark.sql.functions import col, hour, to_date, date_format

# .cast("long") actually count second from 1970, then count different and /60 change to minute
df = df.withColumn(
    "pickup_delay_minutes",
    (col("Picked_Datetime").cast("long") - col("Order_DateTime").cast("long")) / 60
).withColumn(
    "delivery_minutes",
    (col("Delivered_Datetime").cast("long") - col("Picked_Datetime").cast("long")) / 60
).withColumn(
    "order_date",
    to_date(col("Order_DateTime"))
)

df.select(
    "Order_DateTime",
    "Picked_Datetime",
    "Delivered_Datetime",
    "pickup_delay_minutes",
    "delivery_minutes",
    "order_date"
).show(10, truncate=False)

print("row count:", df.count())

+-------------------+-------------------+-------------------+--------------------+----------------+----------+
|Order_DateTime     |Picked_Datetime    |Delivered_Datetime |pickup_delay_minutes|delivery_minutes|order_date|
+-------------------+-------------------+-------------------+--------------------+----------------+----------+
|2022-02-11 08:35:00|2022-02-11 08:40:00|2022-02-11 09:24:00|5.0                 |44.0            |2022-02-11|
|2022-02-11 10:20:00|2022-02-11 10:35:00|2022-02-11 11:13:00|15.0                |38.0            |2022-02-11|
|2022-02-11 13:50:00|2022-02-11 14:05:00|2022-02-11 14:50:00|15.0                |45.0            |2022-02-11|
|2022-02-11 14:45:00|2022-02-11 14:55:00|2022-02-11 15:27:00|10.0                |32.0            |2022-02-11|
|2022-02-11 15:15:00|2022-02-11 15:30:00|2022-02-11 15:47:00|15.0                |17.0            |2022-02-11|
|2022-02-11 16:30:00|2022-02-11 16:35:00|2022-02-11 16:53:00|5.0                 |18.0            |2022-02-11|
|

## summary

In [22]:
from pyspark.sql.functions import avg, countDistinct, expr , round

kpi_summary = df.agg(
    expr("count(*)").alias("total_orders"),
    round(avg("delivery_minutes"),2).alias("avg_delivery_minutes"),
    round(avg("pickup_delay_minutes"),2).alias("avg_pickup_delay_minutes"),
    round(avg("Delivery_person_Ratings"),2).alias("avg_rider_rating"),
    countDistinct("Delivery_person_ID").alias("distinct_delivery_persons")
)

kpi_summary.show(truncate=False)

+------------+--------------------+------------------------+----------------+-------------------------+
|total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|avg_rider_rating|distinct_delivery_persons|
+------------+--------------------+------------------------+----------------+-------------------------+
|16011       |27.47               |9.97                    |4.64            |1320                     |
+------------+--------------------+------------------------+----------------+-------------------------+



## group by city

In [23]:
from pyspark.sql.functions import avg, count

city_analysis = df.groupBy("City").agg(
    count("*").alias("total_orders"),
    round(avg("delivery_minutes"),2).alias("avg_delivery_minutes"),
    round(avg("pickup_delay_minutes"),2).alias("avg_pickup_delay_minutes"),
    round(avg("Delivery_person_Ratings"),2).alias("avg_rider_rating"),
).orderBy("total_orders", ascending=False)

city_analysis.show(truncate=False)

+-------------+------------+--------------------+------------------------+----------------+
|City         |total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|avg_rider_rating|
+-------------+------------+--------------------+------------------------+----------------+
|metropolitian|12020       |27.41               |9.94                    |4.63            |
|urban        |3544        |27.7                |10.09                   |4.68            |
|Unknown      |391         |27.04               |9.95                    |4.68            |
|semi-urban   |56          |26.86               |9.82                    |4.5             |
+-------------+------------+--------------------+------------------------+----------------+



## Top and Bottom 10 Best-Performing Delivery Drivers by Rating

In [27]:
rider_analysis = df.groupBy("Delivery_person_ID").agg(
    count("*").alias("total_orders"),
    round(avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    round(avg("pickup_delay_minutes"), 2).alias("avg_pickup_delay_minutes"),
    round(avg("Delivery_person_Ratings"), 2).alias("avg_rider_rating")
)

top_riders = rider_analysis.orderBy("avg_rider_rating", ascending=False)
worst_riders = rider_analysis.orderBy("avg_rider_rating", ascending=True)

print("Top 10 Best-Performing Delivery Drivers by Rating")
top_riders.show(10, truncate=False)

print("Bottom 10 Lowest-Performing Delivery Drivers by Rating")
worst_riders.show(10, truncate=False)

Top 10 Best-Performing Delivery Drivers by Rating
+------------------+------------+--------------------+------------------------+----------------+
|Delivery_person_ID|total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|avg_rider_rating|
+------------------+------------+--------------------+------------------------+----------------+
|RANCHIRES05DEL02  |12          |28.0                |7.92                    |4.88            |
|INDORES15DEL03    |8           |29.63               |10.63                   |4.86            |
|RANCHIRES19DEL03  |8           |20.13               |10.63                   |4.86            |
|MYSRES11DEL03     |8           |30.38               |10.0                    |4.86            |
|KOCRES13DEL01     |14          |29.93               |8.93                    |4.86            |
|SURRES08DEL03     |8           |31.63               |8.13                    |4.86            |
|MYSRES01DEL03     |6           |24.0                |9.17                   

## Daily trend summary

In [28]:
daily_analysis = df.groupBy("order_date").agg(
    count("*").alias("total_orders"),
    round(avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    round(avg("pickup_delay_minutes"), 2).alias("avg_pickup_delay_minutes")
).orderBy("order_date")

daily_analysis.show(truncate=False)

+----------+------------+--------------------+------------------------+
|order_date|total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|
+----------+------------+--------------------+------------------------+
|2022-02-11|1138        |27.66               |9.97                    |
|2022-02-12|1002        |27.83               |10.13                   |
|2022-02-13|1125        |27.94               |9.84                    |
|2022-02-14|1007        |27.84               |10.03                   |
|2022-02-15|1124        |27.87               |9.93                    |
|2022-02-16|1006        |27.15               |9.92                    |
|2022-02-17|1133        |26.27               |9.89                    |
|2022-02-18|994         |26.93               |9.95                    |
|2022-03-01|1336        |27.56               |10.02                   |
|2022-03-02|1132        |27.87               |9.96                    |
|2022-03-03|1369        |27.26               |10.03             

## Monthly trend summary

In [29]:
from pyspark.sql.functions import date_format, count, avg, round

df = df.withColumn("order_month", date_format("Order_DateTime", "yyyy-MM"))

monthly_trend = df.groupBy("order_month").agg(
    count("*").alias("total_orders"),
    round(avg("delivery_minutes"), 2).alias("avg_delivery_minutes"),
    round(avg("pickup_delay_minutes"), 2).alias("avg_pickup_delay_minutes"),
    round(avg("Delivery_person_Ratings"), 2).alias("avg_rider_rating")
).orderBy("order_month")

monthly_trend.show(truncate=False)

+-----------+------------+--------------------+------------------------+----------------+
|order_month|total_orders|avg_delivery_minutes|avg_pickup_delay_minutes|avg_rider_rating|
+-----------+------------+--------------------+------------------------+----------------+
|2022-02    |8529        |27.44               |9.96                    |4.64            |
|2022-03    |7482        |27.5                |9.99                    |4.64            |
+-----------+------------+--------------------+------------------------+----------------+



## save to csv (summary and all group info)